# Initial SoC Dataset Construction

## Objective
Construct datasets for **Initial State of Charge (SoC) Estimation**.

**Synthetic-Initialization Definition:**
> "At an arbitrary valid point $t_i$ along a recorded battery trajectory, we simulate the estimator becoming active with unknown SoC. The estimator observes V, I, and battery temperature for a finite observation duration and estimates the SoC at the beginning of that observation window. These are synthetic initialization windows, not literal recorded power-on events."

**Problem Formulation:**
- **Input:** $X_i = [V_{i...i+N-1}, I_{i...i+N-1}, T_{i...i+N-1}]$ (for a fixed observation duration)
- **Target:** $y_i = SoC_{ref}(i)$ (the SoC at the START of the window)

**Operating-Condition & Aging Generalization:**
The Panasonic test campaign used the same physical 18650PF cell tested sequentially: 25°C → 10°C → 0°C → -10°C → -20°C. 
- The Test set (-20°C) is both the coldest temperature AND the most degraded state of the battery (capacity faded from 2.8 Ah to ~2.3 Ah).
- Generalization is explicitly entangled between temperature and aging. This is not pure cross-cell temperature generalization.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io as sio
from pathlib import Path
import os

def _find_project_root():
    """Locate this repository from its root or notebooks working directory."""
    cwd = Path.cwd().resolve()
    candidates = (cwd, cwd.parent) if cwd.name == "notebooks" else (cwd,)
    for candidate in candidates:
        if ((candidate / "AGENTS.md").is_file()
                and (candidate / "src" / "data_loader.py").is_file()):
            return candidate
    raise RuntimeError(
        "Cannot locate project repository: launch from its root or notebooks/ "
        "(requires AGENTS.md and src/data_loader.py)."
    )


PROJECT_ROOT = _find_project_root()
DATASET_ROOT = PROJECT_ROOT / "DATASET/Panasonic/Panasonic 18650PF Data"
RESULTS_DIR  = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)



## 1. Valid File Selection & Leakage-Free Splitting
We select only individual dynamic drive-cycle trajectories.
Explicitly combined/duplicate files are excluded. Splitting is strictly per-file to prevent target leakage.


In [ ]:
# Define exact candidate directories
CANDIDATE_PATHS = {
    '25degC': DATASET_ROOT / "Panasonic 18650PF Data/25degC/Drive cycles",
    '10degC': DATASET_ROOT / "10degC/Drive Cycles",
    '0degC':  DATASET_ROOT / "0degC/Drive cycles",
    '-10degC': DATASET_ROOT / "-10degC/Drive Cycles",
    '-20degC': DATASET_ROOT / "-20degC/Drive cycles"
}

splits = {
    'train': [],
    'val': [],
    'test': []
}

for temp, folder in CANDIDATE_PATHS.items():
    if temp in ['25degC', '10degC', '0degC']:
        split_key = 'train'
    elif temp == '-10degC':
        split_key = 'val'
    else:
        split_key = 'test'
        
    for f in sorted(folder.glob('*.mat')):
        fname_lower = f.name.lower()
        match_count = sum(1 for kw in ['hwfet', 'udds', 'la92', 'nn', 'us06'] if kw in fname_lower)
        if match_count > 1:
            print(f"Excluding combined/duplicate file: {f.name}")
        else:
            splits[split_key].append(f)

print("\n=== File Split Distribution ===")
for split_name, files in splits.items():
    print(f"{split_name.capitalize()}: {len(files)} files")



## 2. Window Generation Logic
- **Features:** `[Voltage_V, Current_A, Battery_Temp_C]`
- **Target:** `SoC_ref` at window start.
- **Stride:** 500 samples (~50 seconds). Provides non-overlapping windows for the 50s duration case. Samples from the same trajectory are correlated, but this maximizes unique coverage without overlap.
- **Validation:** 
  1. Internal gap check: reject if `max(dt) > 0.5s` inside window.
  2. Duration check: reject if total duration deviates > 20% from target.
  3. No NaNs.



In [ ]:
def extract_windows(file_path, target_duration_s, stride=500):
    """Extract initial-SoC windows from a single trajectory file."""
    mat = sio.loadmat(file_path)
    meas = mat['meas']
    
    time = meas['Time'][0, 0].flatten()
    volt = meas['Voltage'][0, 0].flatten()
    curr = meas['Current'][0, 0].flatten()
    temp = meas['Battery_Temp_degC'][0, 0].flatten()
    ah   = meas['Ah'][0, 0].flatten()
    
    # Calculate Ah-based Reference SoC
    soc_ref = np.clip(((2.9 + ah) / 2.9) * 100.0, 0.0, 100.0)
    
    # Assume 10 Hz nominal sampling for required sample count
    req_samples = int(target_duration_s * 10)
    
    X_list, y_list, t_list = [], [], []
    stats = {'candidate': 0, 'rej_dur': 0, 'rej_gap': 0, 'rej_nan': 0, 'accepted': 0}
    
    max_idx = len(time) - req_samples
    if max_idx <= 0:
        return X_list, y_list, t_list, stats
        
    # Generate start indices
    start_indices = list(range(0, max_idx + 1, stride))
    stats['candidate'] = len(start_indices)
    
    for i in start_indices:
        idx_end = i + req_samples
        
        window_time = time[i:idx_end]
        actual_dur = window_time[-1] - window_time[0]
        
        # 1. Duration check (+/- 20%)
        dur_tolerance = 0.20 * target_duration_s
        if abs(actual_dur - target_duration_s) > dur_tolerance:
            stats['rej_dur'] += 1
            continue
            
        # 2. Internal gap check (dt > 0.5s is anomalous)
        dt_max = np.diff(window_time).max()
        if dt_max > 0.5:
            stats['rej_gap'] += 1
            continue
            
        # 3. Extract features
        x_window = np.column_stack([
            volt[i:idx_end],
            curr[i:idx_end],
            temp[i:idx_end]
        ])
        
        if np.isnan(x_window).any():
            stats['rej_nan'] += 1
            continue
            
        # 4. Target is at the START of the window
        y_target = soc_ref[i]
        
        X_list.append(x_window)
        t_list.append(window_time - window_time[0])
        y_list.append(y_target)
        stats['accepted'] += 1
        
    return X_list, y_list, t_list, stats




In [ ]:
# Generate datasets for all durations
DURATIONS = [5, 10, 20, 30, 50, 100]
STRIDE = 500

datasets = {}
all_stats = {}

print("=== Generating Observation Windows ===\n")

for dur in DURATIONS:
    print(f"--- Target Duration: {dur} seconds ({dur*10} samples) ---")
    datasets[dur] = {}
    
    total_stats = {'candidate': 0, 'rej_dur': 0, 'rej_gap': 0, 'rej_nan': 0, 'accepted': 0}
    
    # Store aggregate SoC to report distribution
    y_all = []
    
    for split_name, files in splits.items():
        X_split, y_split, t_split = [], [], []
        
        for f in files:
            X_f, y_f, t_f, s = extract_windows(f, dur, STRIDE)
            X_split.extend(X_f)
            y_split.extend(y_f)
            t_split.extend(t_f)
            y_all.extend(y_f)
            
            for k in total_stats:
                total_stats[k] += s[k]
                
        datasets[dur][f'X_{split_name}'] = np.array(X_split, dtype=np.float32)
        datasets[dur][f'y_{split_name}'] = np.array(y_split, dtype=np.float32)
        datasets[dur][f't_{split_name}'] = np.array(t_split, dtype=np.float32)
        
    all_stats[dur] = total_stats
    
    print(f"Candidate starts: {total_stats['candidate']:>6,}")
    print(f"Rejected (dur)  : {total_stats['rej_dur']:>6,}")
    print(f"Rejected (gap)  : {total_stats['rej_gap']:>6,}")
    print(f"Rejected (NaN)  : {total_stats['rej_nan']:>6,}")
    print(f"Final Accepted  : {total_stats['accepted']:>6,}")
    
    # SoC Distribution for this duration
    y_arr = np.array(y_all)
    if len(y_arr) > 0:
        print(f"SoC Dist -> Min:{y_arr.min():.1f}%, Max:{y_arr.max():.1f}%, Mean:{y_arr.mean():.1f}%, Std:{y_arr.std():.1f}%")
        # 10% bins
        counts, _ = np.histogram(y_arr, bins=np.arange(0, 110, 10))
        print("SoC Bins (0-10, 10-20...): " + ", ".join(f"{c/len(y_arr)*100:.0f}%" for c in counts))
    print()




## 3. Scaling
Scale features globally based on the training files. We will use the 50s duration Train set to compute the Min/Max, and apply that consistent scaler to all durations.
*Features are expected to be strictly: [Voltage_V, Current_A, Battery_Temp_C]*


In [ ]:
# Compute global Min/Max from Train set (using 50s windows)
X_train_50 = datasets[50]['X_train']

# Reshape to (N*steps, 3) to find global min/max
X_train_flat = X_train_50.reshape(-1, 3)
feat_min = X_train_flat.min(axis=0)
feat_max = X_train_flat.max(axis=0)
feat_range = feat_max - feat_min

print("Scaler computed from Train split:")
print(f"Voltage (V) : min={feat_min[0]:.4f}, max={feat_max[0]:.4f}")
print(f"Current (A) : min={feat_min[1]:.4f}, max={feat_max[1]:.4f}")
print(f"Temp (°C)   : min={feat_min[2]:.4f}, max={feat_max[2]:.4f}")

# Apply normalization across ALL durations and splits
for dur in DURATIONS:
    for split_name in ['train', 'val', 'test']:
        key = f'X_{split_name}'
        if len(datasets[dur][key]) > 0:
            datasets[dur][key] = (datasets[dur][key] - feat_min) / feat_range

# Normalize Target (y)
target_min = 0.0
target_max = 100.0

for dur in DURATIONS:
    for split_name in ['train', 'val', 'test']:
        key = f'y_{split_name}'
        if len(datasets[dur][key]) > 0:
            datasets[dur][key] = (datasets[dur][key] - target_min) / 100.0




## 4. Sanity Check (Explicit Output)


In [ ]:
# Print an exact example from the 20s test set
sample_x = datasets[20]['X_test'][0]  # First window of validation
sample_y = datasets[20]['y_test'][0]

# Denormalize to display physical values
phys_x = sample_x * feat_range + feat_min
phys_y = sample_y * 100.0

print("=== Example 20-second Window (Test Set) ===")
print(f"Shape: {sample_x.shape}")
print(f"Target Initial SoC: {phys_y:.2f}%")
print(f"First Sample (t=0): V={phys_x[0,0]:.3f}V, I={phys_x[0,1]:.3f}A, T={phys_x[0,2]:.1f}°C")
print(f"Last Sample  (t=T): V={phys_x[-1,0]:.3f}V, I={phys_x[-1,1]:.3f}A, T={phys_x[-1,2]:.1f}°C")



## 5. Save Final Artifacts


In [ ]:
import pickle

out_file = PROJECT_ROOT / "data" / "initial_soc_datasets.pkl"
out_file.parent.mkdir(exist_ok=True)

# Save dict of dictionaries
with open(out_file, 'wb') as f:
    pickle.dump(datasets, f)

# Save scaler
scaler_file = PROJECT_ROOT / "data" / "initial_soc_scaler.npz"
np.savez_compressed(
    scaler_file,
    feat_min=feat_min,
    feat_range=feat_range,
    target_min=target_min,
    target_range=100.0
)

print(f"Saved datasets to: {out_file}")
print(f"Saved scaler to: {scaler_file}")

